## connection MYSQl

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

## Library Analisis

In [ ]:
import pandas as pd;
pd.set_option('display.max_rows', None)

## Mahasiswa

In [ ]:
prodiQuery = """
SELECT uuid,name FROM prodi
"""

prodiDf = pd.read_sql(prodiQuery, siakadEngine)

prodiDf

In [ ]:
jenisKeluarQuery = """
SELECT code,name FROM jenis_keluar where is_mahasiswa = 1
""";

jenisKeluarDf = pd.read_sql(jenisKeluarQuery, siakadEngine);

jenisKeluarDf

### Jumlah Mahasiwa Aktif Per Tahun dan Program Studi

In [ ]:
mahasiswaAktifQuery = """
SELECT
    semesters.year_of_study AS tahun_angkatan,
    prodi.name AS program_studi,
    COUNT(registers.registration_uuid) as mahasiswa_aktif

FROM mahasiswa_registers AS registers

JOIN semesters ON 
    registers.semester_code = semesters.code

JOIN prodi ON 
    registers.prodi_uuid = prodi.uuid

WHERE
    semesters.year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
    registers.id_jenis_keluar IS NULL AND
    registers.is_external = 0 AND
    registers.is_archive = 0 AND
    registers.deleted_at IS NULL
GROUP BY
    semesters.year_of_study,
    prodi.name
ORDER BY semesters.year_of_study DESC
"""

In [ ]:
mahasiwaAktifDf = pd.read_sql(mahasiswaAktifQuery, siakadEngine);

In [ ]:
mahasiwaAktifDf

### Validasi Perhitungan Mahasiswa Aktif

In [ ]:
validasiMahasiswaAktifQuery = text("""
SELECT
    registers.nim,
    details.name as nama_mahasiswa,
    prodi.name AS program_study,
    semesters.year_of_study AS tahun_angkatan

FROM mahasiswa_registers AS registers

JOIN mahasiswa_details AS details ON
    registers.mahasiswa_uuid = details.uuid

JOIN prodi ON
    registers.prodi_uuid = prodi.uuid

JOIN semesters ON 
    registers.semester_code = semesters.code

WHERE
    registers.prodi_uuid = :prodi_uuid AND
    semesters.year_of_study = :year_of_study AND
    registers.id_jenis_keluar IS NULL AND
    registers.is_external = 0 AND
    registers.is_archive = 0 AND
    registers.deleted_at IS NULL
""")

parameter_query = {
    "prodi_uuid": "b6462f14-62ca-48e9-ada2-33b47cbe9246",
    "year_of_study": 2022
}

validasiMahasiswaDf = pd.read_sql(
    validasiMahasiswaAktifQuery, 
    siakadEngine, 
    params=parameter_query
)

total_data = len(validasiMahasiswaDf)
print(f"Total data mahasiswa: {total_data}")

validasiMahasiswaDf

### Eksplorasi Data

In [ ]:

summary_list = []

sumaryDf = jenisKeluarDf

for col in sumaryDf.select_dtypes(include=['object', 'category', 'int64']).columns:
    vc = sumaryDf[col].value_counts(dropna=False).head(10).reset_index()
    vc.columns = ['Nilai Data', 'Jumlah']
    vc.insert(0, 'Nama Kolom', col)
    
    summary_list.append(vc)

final_summary = pd.concat(summary_list, ignore_index=True)

final_summary